# 10 — Dispatch Patterns: Synchronous and Asynchronous

This notebook demonstrates how the holonic framework supports different dispatch patterns against the same underlying API. The framework's public surface is synchronous by design (see SPEC R2.5) — `ds.traverse()`, `ds.add_holon()`, `ds.validate_membrane()` all block and return. But the choice of *how* to drive those calls is up to the caller, and the framework accommodates several disciplines.

This is the same architectural split the W3C DOM uses: a synchronous API layered over a substrate that may itself be asynchronous. See `docs/source/dom-comparison.md` for the full mapping.

Three patterns covered:

1. **Direct synchronous dispatch** — the caller invokes holarchy operations in a sequential loop. Analog: `element.click()` in a script.
2. **Event-queue dispatch** — the caller enqueues dispatch requests and drains them via a worker. The caller returns immediately; the worker processes the queue on its own schedule. Analog: React's reconciliation queue flushing to the real DOM.
3. **Asyncio dispatch over an executor** — the caller uses `asyncio.run_in_executor` to offload blocking traversals, getting an awaitable interface over the synchronous API. Analog: the browser's JavaScript engine wrapping synchronous DOM operations in a promise-like API for callers that want to chain them.

All three operate against the same `HolonicDataset`. The framework doesn't know or care which one the caller chooses.

## A note on concurrency and rdflib

The in-memory `RdflibBackend` is **not thread-safe for concurrent SPARQL parsing**. rdflib uses pyparsing internally, and pyparsing's parser state is shared across threads. Two threads parsing SPARQL simultaneously produce undefined behavior, typically a `TypeError` deep in pyparsing internals.

This doesn't preclude asynchronous dispatch patterns — it means that against the in-memory backend, traversals must still serialize through the library. Against an HTTP backend like Fuseki, true parallelism is available because the parsing happens server-side and each HTTP request is independent.

The patterns below are written to work correctly with the in-memory backend. The event-queue pattern uses a single worker thread (serialized by construction). The asyncio pattern uses a single-worker executor. Against Fuseki, both patterns can be relaxed to use real parallelism.

## Setup

A small holarchy with three downstream holons, each with a boundary shape and a portal from a common upstream source. We'll dispatch traversals across all three under each pattern.

In [ ]:
from holonic import HolonicDataset

ds = HolonicDataset()

# Source holon with a track observation
ds.add_holon("urn:holon:source", "Source")
ds.add_interior("urn:holon:source", '''
    @prefix src: <urn:src:> .
    <urn:track:001> a src:Track ;
        src:latitude 34.05 ;
        src:longitude -118.25 ;
        src:confidence 0.92 .
''')

# Three downstream holons, each reshaping the source data
for name in ("analyst", "display", "audit"):
    ds.add_holon(f"urn:holon:{name}", name.capitalize())
    ds.add_boundary(f"urn:holon:{name}", '''
        @prefix sh: <http://www.w3.org/ns/shacl#> .
        @prefix dst: <urn:dst:> .
        [] a sh:NodeShape ;
            sh:targetClass dst:Observation ;
            sh:property [ sh:path dst:location ; sh:minCount 1 ] ;
            sh:property [ sh:path dst:score ; sh:minCount 1 ] .
    ''')

# One CONSTRUCT portal from source to each downstream
portal_cq = '''
    PREFIX src: <urn:src:>
    PREFIX dst: <urn:dst:>
    CONSTRUCT {
        ?t a dst:Observation ;
            dst:location ?loc ;
            dst:score ?conf .
    }
    WHERE {
        GRAPH ?g {
            ?t a src:Track ;
                src:latitude ?lat ;
                src:longitude ?lon ;
                src:confidence ?conf .
            BIND(CONCAT(STR(?lat), ",", STR(?lon)) AS ?loc)
        }
    }
'''

for name in ("analyst", "display", "audit"):
    ds.add_portal(
        f"urn:portal:source-to-{name}",
        source_iri="urn:holon:source",
        target_iri=f"urn:holon:{name}",
        construct_query=portal_cq,
        label=f"source to {name}",
    )

print(f"Holons: {len(ds.list_holons())}")
print(f"Portals: {len(ds.list_portals())}")

## Pattern 1: Direct Synchronous Dispatch

The simplest pattern. The caller invokes `traverse()` once per target, in sequence. Each call returns before the next begins. Analogous to a script calling `element.click()` in a tight loop — the DOM updates after each call, with no concurrency.

Use when: the holarchy is small, the backend is in-memory, and the caller is not bottlenecked on any single traversal.

In [ ]:
import time

def dispatch_sync(ds, targets):
    """Direct sequential dispatch. Each traversal blocks until complete."""
    results = []
    for target in targets:
        projected, membrane = ds.traverse(
            source_iri="urn:holon:source",
            target_iri=target,
            validate=True,
            agent_iri="urn:agent:sync-dispatcher",
        )
        results.append((target, len(projected), membrane.health.value))
    return results

targets = [
    "urn:holon:analyst",
    "urn:holon:display",
    "urn:holon:audit",
]

start = time.perf_counter()
results = dispatch_sync(ds, targets)
elapsed_sync = time.perf_counter() - start

print(f"Synchronous dispatch: {elapsed_sync*1000:.1f}ms")
for target, triples, health in results:
    print(f"  {target}: {triples} triples, membrane={health}")

## Pattern 2: Event-Queue Dispatch

A structured asynchronous pattern: dispatch requests enter a queue, and a single worker thread drains the queue at its own pace. The caller never blocks on a traversal — it enqueues the request and moves on.

This matches React's reconciliation pattern: state updates are enqueued, batched, and flushed to the real DOM when the scheduler decides. The DOM itself doesn't change; React's layer adds queuing and scheduling on top.

Because the worker is a single thread, SPARQL parsing happens serially — safe against the in-memory rdflib backend. Against an HTTP backend you could scale up to multiple workers.

In [ ]:
import queue
import threading

class HolonicDispatcher:
    """Minimal event-queue dispatcher over a HolonicDataset.

    Dispatch requests enqueue and return immediately. A single worker
    thread drains the queue, invoking ds.traverse() synchronously.
    Results collect in a dict keyed by target IRI.

    Single-worker by design: against the in-memory rdflib backend,
    concurrent SPARQL parsing is not safe, so the queue pattern
    naturally serializes. Against Fuseki the worker count can be
    increased.
    """

    def __init__(self, ds, agent_iri="urn:agent:queue-dispatcher"):
        self._ds = ds
        self._agent = agent_iri
        self._queue = queue.Queue()
        self._results = {}
        self._stopping = threading.Event()
        self._worker = threading.Thread(target=self._run, daemon=True)
        self._worker.start()

    def _run(self):
        while not self._stopping.is_set():
            try:
                job = self._queue.get(timeout=0.1)
            except queue.Empty:
                continue
            source, target = job
            try:
                projected, membrane = self._ds.traverse(
                    source_iri=source,
                    target_iri=target,
                    validate=True,
                    agent_iri=self._agent,
                )
                self._results[target] = (len(projected), membrane.health.value)
            finally:
                self._queue.task_done()

    def submit(self, source, target):
        """Enqueue a traversal. Returns immediately."""
        self._queue.put((source, target))

    def wait(self):
        """Block until all queued traversals complete."""
        self._queue.join()

    def stop(self):
        self._stopping.set()
        self._worker.join(timeout=1.0)

    @property
    def results(self):
        return dict(self._results)


dispatcher = HolonicDispatcher(ds)

start = time.perf_counter()
for target in targets:
    dispatcher.submit("urn:holon:source", target)
    # Caller continues immediately; traversal runs on the worker

# Simulate doing other work while the queue drains
time.sleep(0.005)

dispatcher.wait()
elapsed_queue = time.perf_counter() - start
dispatcher.stop()

print(f"Queue dispatch: {elapsed_queue*1000:.1f}ms (submission + simulated work + drain)")
for target, (triples, health) in dispatcher.results.items():
    print(f"  {target}: {triples} triples, membrane={health}")

## Pattern 3: Asyncio Dispatch over an Executor

The synchronous traverse API can be lifted into an `async def` interface using `asyncio.get_event_loop().run_in_executor()`. Callers can then `await ds.dispatch_async(...)` and compose traversals with other asyncio work (HTTP servers, streaming clients, scheduled tasks).

The executor is a single-worker `ThreadPoolExecutor`. `max_workers=1` serializes by construction — only one traversal runs at a time, which keeps rdflib happy.

This is the pattern most callers will reach for when integrating holonic into an async application (FastAPI endpoint, aiohttp handler, asyncio-based agent runtime).

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

class AsyncHolonicAdapter:
    """Lift the synchronous HolonicDataset API into an awaitable one.

    Single-worker executor serializes access to the backend, which is
    necessary for in-memory rdflib. The caller sees an async interface
    and can compose traversals with other asyncio work.
    """

    def __init__(self, ds, agent_iri="urn:agent:asyncio-dispatcher"):
        self._ds = ds
        self._agent = agent_iri
        self._executor = ThreadPoolExecutor(max_workers=1)

    async def traverse(self, source, target):
        loop = asyncio.get_running_loop()

        def _blocking():
            return self._ds.traverse(
                source_iri=source,
                target_iri=target,
                validate=True,
                agent_iri=self._agent,
            )

        projected, membrane = await loop.run_in_executor(self._executor, _blocking)
        return target, len(projected), membrane.health.value

    def close(self):
        self._executor.shutdown(wait=True)


async def run_async_dispatch():
    adapter = AsyncHolonicAdapter(ds)
    try:
        # Fire all traversals concurrently from the caller's perspective.
        # The executor serializes them, so they complete in submission order
        # against the in-memory backend; against Fuseki, asyncio.gather could
        # genuinely parallelize I/O-bound waits.
        start = time.perf_counter()
        coros = [adapter.traverse("urn:holon:source", t) for t in targets]
        results = await asyncio.gather(*coros)
        elapsed = time.perf_counter() - start
        return elapsed, results
    finally:
        adapter.close()

# Jupyter supports top-level await in notebook cells.
# In a regular Python script, wrap this with: asyncio.run(run_async_dispatch())
elapsed_async, async_results = await run_async_dispatch()

print(f"Asyncio dispatch: {elapsed_async*1000:.1f}ms")
for target, triples, health in async_results:
    print(f"  {target}: {triples} triples, membrane={health}")

## Provenance Records Preserved Across Patterns

All three dispatch patterns write identical PROV-O records to each target's context graph. The `prov:wasAssociatedWith` value differs (each dispatcher uses a different agent IRI), but the structure of the activity record, the validation outcome, and the traversal timing are captured the same way.

This is the key guarantee: *how* you dispatch doesn't change *what* the framework records. PROV-O remains the canonical audit trail regardless of dispatch discipline.

In [ ]:
# Count activities on urn:holon:analyst by dispatcher agent
rows = ds.query('''
    PREFIX prov: <http://www.w3.org/ns/prov#>

    SELECT ?agent (COUNT(?activity) AS ?n_activities)
    WHERE {
        GRAPH <urn:holon:analyst/context> {
            ?activity a prov:Activity ;
                prov:wasAssociatedWith ?agent .
        }
    }
    GROUP BY ?agent
    ORDER BY ?agent
''')

print("Activities on urn:holon:analyst by dispatcher agent:")
for row in rows:
    print(f"  {row['agent']}: {row['n_activities']} activities")

## Choosing a Pattern

| Pattern | Best for | Tradeoffs |
|---------|---------|-----------|
| Synchronous | Simple scripts, CLI tools, small holarchies | Blocks the caller on every traversal |
| Event queue | Non-blocking callers, batching, rate limiting | Caller can't observe individual completion inline |
| Asyncio | Integration into async apps (FastAPI, agents) | Serialization still required against in-memory backend |

Choose based on where the bottleneck actually is and what the consumer needs:

- **CPU-bound operations against in-memory rdflib** — threading and asyncio don't help. The GIL serializes threads, and the single-worker executor pattern above is explicit about it.
- **Network-bound operations against Fuseki or another HTTP store** — concurrency unlocks meaningful throughput. Multiple workers in the queue pattern or `max_workers > 1` in the asyncio pattern becomes safe and useful.
- **Integration with async applications** — the asyncio adapter is the right fit even when it doesn't improve performance, because it lets holonic participate in the application's existing async flow.

The framework accommodates all three patterns without change. That flexibility is load-bearing for the "holonic as stage, orchestrators as players" architectural stance discussed in `docs/source/dom-comparison.md`.

## Where This Does Not (Yet) Apply

The framework has no asynchronous API surface as of 0.4.1. There is no `await ds.traverse(...)` natively — callers who need async integration build it on top of the synchronous API using patterns like `AsyncHolonicAdapter` above.

Whether to add a parallel async protocol (`AsyncHolonicStore`) is captured as an open question in SPEC R2.5. The current commitment: the sync protocol is canonical; an async variant, if added, will be a separate protocol rather than a replacement.

See also:

- `docs/source/dom-comparison.md` — detailed treatment of sync API over async substrate
- `docs/SPEC.md` R2.5 — the sync-protocol commitment
- `docs/SPEC.md` OQ9 — DOM-style event propagation as a coordination model